Task 1: Dataset Understanding

In [1]:
import pandas as pd

# Load the data
df = pd.read_csv('customer_churn_nn.csv')

# Inspect data
print("--- HEAD ---")
print(df.head())
print("\n--- INFO ---")
df.info()
print("\n--- SHAPE ---")
print(df.shape)

--- HEAD ---
  customer_id   region plan_type   contract_type payment_method  \
0    CUST0001    South  Standard  Month-to-month     Debit Card   
1    CUST0002     West   Premium  Month-to-month         Wallet   
2    CUST0003  Central  Standard  Month-to-month    Credit Card   
3    CUST0004     West   Premium  Month-to-month    Credit Card   
4    CUST0005    North   Premium  Month-to-month    Net Banking   

   tenure_months  monthly_charges_inr  avg_login_days_per_month  \
0             30               687.40                        13   
1             15              1029.74                        22   
2             72               732.07                        13   
3             22               959.51                        19   
4             11               890.20                        18   

   support_tickets_last_90_days  payment_delay_days  data_usage_gb  \
0                             0                   0          87.97   
1                             3          

Task 2: Data Preprocessing

In [2]:
# Missing value check
missing_values = df.isnull().sum()
print("--- MISSING VALUES ---")
print(missing_values)

# Distribution of the target variable
churn_dist = df['churn'].value_counts(normalize=True)
churn_counts = df['churn'].value_counts()
print("\n--- CHURN DISTRIBUTION ---")
print("Counts:")
print(churn_counts)
print("Proportions:")
print(churn_dist)

# Basic statistical summary
print("\n--- STATISTICAL SUMMARY ---")
print(df.describe())

--- MISSING VALUES ---
customer_id                     0
region                          0
plan_type                       0
contract_type                   0
payment_method                  0
tenure_months                   0
monthly_charges_inr             0
avg_login_days_per_month        0
support_tickets_last_90_days    0
payment_delay_days              0
data_usage_gb                   0
satisfaction_score              0
last_complaint_days_ago         0
discount_percent                0
autopay_enabled                 0
referral_count                  0
churn                           0
dtype: int64

--- CHURN DISTRIBUTION ---
Counts:
churn
0    1969
1      31
Name: count, dtype: int64
Proportions:
churn
0    0.9845
1    0.0155
Name: proportion, dtype: float64

--- STATISTICAL SUMMARY ---
       tenure_months  monthly_charges_inr  avg_login_days_per_month  \
count    2000.000000          2000.000000               2000.000000   
mean       25.362000           766.487295          

Task 3: Neural Network Model Building

In [3]:
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

region: <StringArray>
['South', 'West', 'Central', 'North', 'East']
Length: 5, dtype: str
plan_type: <StringArray>
['Standard', 'Premium', 'Basic', 'Enterprise']
Length: 4, dtype: str
contract_type: <StringArray>
['Month-to-month', 'One-year', 'Two-year']
Length: 3, dtype: str
payment_method: <StringArray>
['Debit Card', 'Wallet', 'Credit Card', 'Net Banking', 'UPI']
Length: 5, dtype: str


In [ ]:
##!pip install tensorflow

In [6]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [ ]:
##!pip install scikit-learn

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Load dataset
df = pd.read_csv('customer_churn_nn.csv')

# Drop customer_id
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

# One-hot encode categorical features
X = pd.get_dummies(X, columns=['region', 'plan_type', 'contract_type', 'payment_method'], drop_first=True)

# Convert boolean columns to int if get_dummies produced bools
for col in X.columns:
    if X[col].dtype == bool:
        X[col] = X[col].astype(int)

# Split the data into train and test sets (80/20 split) with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Identify numerical columns to scale (columns that are not binary dummy variables)
# Let's see the columns
num_cols = ['tenure_months', 'monthly_charges_inr', 'avg_login_days_per_month', 
            'support_tickets_last_90_days', 'payment_delay_days', 'data_usage_gb', 
            'satisfaction_score', 'last_complaint_days_ago', 'discount_percent', 'referral_count']

# Scale numerical columns
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# Convert to float32 numpy arrays for TensorFlow
X_train_arr = X_train_scaled.values.astype(np.float32)
X_test_arr = X_test_scaled.values.astype(np.float32)
y_train_arr = y_train.values.astype(np.float32)
y_test_arr = y_test.values.astype(np.float32)

print("Preprocessed X_train shape:", X_train_arr.shape)
print("Preprocessed X_test shape:", X_test_arr.shape)

Preprocessed X_train shape: (1600, 24)
Preprocessed X_test shape: (400, 24)


In [11]:
def run_experiment(name, hidden_layers, lr, batch_size, epochs, activation='relu'):
    tf.keras.backend.clear_session()
    tf.random.set_seed(42)
    np.random.seed(42)
    
    model = Sequential()
    # Input layer and first hidden layer
    model.add(Dense(hidden_layers[0], input_dim=X_train_arr.shape[1], activation=activation))
    
    # Additional hidden layers
    for units in hidden_layers[1:]:
        model.add(Dense(units, activation=activation))
        
    # Output layer
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train model
    history = model.fit(X_train_arr, y_train_arr, epochs=epochs, batch_size=batch_size, 
                        validation_data=(X_test_arr, y_test_arr), verbose=0)
    
    # Evaluate model
    train_loss, train_acc = model.evaluate(X_train_arr, y_train_arr, verbose=0)
    test_loss, test_acc = model.evaluate(X_test_arr, y_test_arr, verbose=0)
    
    # Predictions
    y_pred_prob = model.predict(X_test_arr, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Metrics
    cm = confusion_matrix(y_test_arr, y_pred)
    prec = precision_score(y_test_arr, y_pred, zero_division=0)
    rec = recall_score(y_test_arr, y_pred, zero_division=0)
    f1 = f1_score(y_test_arr, y_pred, zero_division=0)
    
    results = {
        'Experiment Name': name,
        'Layers/Neurons': str(hidden_layers),
        'Activation': activation,
        'Learning Rate': lr,
        'Batch Size': batch_size,
        'Epochs': epochs,
        'Train Loss': round(train_loss, 4),
        'Train Accuracy': round(train_acc, 4),
        'Test Loss': round(test_loss, 4),
        'Test Accuracy': round(test_acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'Confusion Matrix': str(cm.tolist())
    }
    return results, history, model

# Let's test the baseline model first
baseline_res, baseline_hist, baseline_model = run_experiment(
    name='Baseline (Exp 0)', 
    hidden_layers=[32], 
    lr=0.001, 
    batch_size=32, 
    epochs=50, 
    activation='relu'
)

print(baseline_res)

c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


{'Experiment Name': 'Baseline (Exp 0)', 'Layers/Neurons': '[32]', 'Activation': 'relu', 'Learning Rate': 0.001, 'Batch Size': 32, 'Epochs': 50, 'Train Loss': 0.0476, 'Train Accuracy': 0.985, 'Test Loss': 0.0554, 'Test Accuracy': 0.9875, 'Precision': 1.0, 'Recall': 0.1667, 'F1-Score': 0.2857, 'Confusion Matrix': '[[394, 0], [5, 1]]'}


In [12]:
experiments = []

# Baseline
res0, _, _ = run_experiment('Baseline (Exp 0)', [32], 0.001, 32, 50, 'relu')
experiments.append(res0)

# Experiment 1: More layers & neurons
res1, _, _ = run_experiment('Experiment 1: More Layers', [64, 32], 0.001, 32, 50, 'relu')
experiments.append(res1)

# Experiment 2: Higher Learning Rate
res2, _, _ = run_experiment('Experiment 2: High LR', [32], 0.01, 32, 50, 'relu')
experiments.append(res2)

# Experiment 3: Different Activation & More Epochs
res3, _, _ = run_experiment('Experiment 3: Tanh & Epochs', [32], 0.001, 64, 100, 'tanh')
experiments.append(res3)

# Combine into a DataFrame
exp_df = pd.DataFrame(experiments)
print(exp_df.to_string())

c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer

               Experiment Name Layers/Neurons Activation  Learning Rate  Batch Size  Epochs  Train Loss  Train Accuracy  Test Loss  Test Accuracy  Precision  Recall  F1-Score    Confusion Matrix
0             Baseline (Exp 0)           [32]       relu          0.001          32      50      0.0476          0.9850     0.0517         0.9875     1.0000  0.1667    0.2857  [[394, 0], [5, 1]]
1    Experiment 1: More Layers       [64, 32]       relu          0.001          32      50      0.0034          1.0000     0.0953         0.9850     0.0000  0.0000    0.0000  [[394, 0], [6, 0]]
2        Experiment 2: High LR           [32]       relu          0.010          32      50      0.0011          1.0000     0.1613         0.9825     0.3333  0.1667    0.2222  [[392, 2], [5, 1]]
3  Experiment 3: Tanh & Epochs           [32]       tanh          0.001          64     100      0.0483          0.9862     0.0547         0.9850     0.0000  0.0000    0.0000  [[394, 0], [6, 0]]


In [13]:
# Let's save the preprocessed training dataset to a CSV file
train_preprocessed = pd.DataFrame(X_train_arr, columns=X.columns)
train_preprocessed['churn'] = y_train_arr.astype(int)
train_preprocessed.to_csv('customer_churn_preprocessed_train.csv', index=False)
print("Saved preprocessed training data to customer_churn_preprocessed_train.csv")

Saved preprocessed training data to customer_churn_preprocessed_train.csv


In [ ]:
##!pip install matplotlib

In [16]:
import matplotlib.pyplot as plt

# Let's get history of Experiment 1 since it shows some interesting learning behavior
_, exp1_hist, _ = run_experiment('Experiment 1: More Layers', [64, 32], 0.001, 32, 50, 'relu')

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(exp1_hist.history['loss'], label='Train Loss')
plt.plot(exp1_hist.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(exp1_hist.history['accuracy'], label='Train Acc')
plt.plot(exp1_hist.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig('learning_curves.png')
plt.close()
print("Saved learning curves plot to learning_curves.png")

Matplotlib is building the font cache; this may take a moment.
c:\Users\nehaf\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Saved learning curves plot to learning_curves.png
